In [7]:
# Mount to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Set Dataset File Path (Google Drive)
zip_path = '/content/drive/MyDrive/Deep Learning/Small_Pop_Song_Dataset.zip'
dest_folder = '/content/drive/MyDrive/Deep Learning'

!unzip -q "$zip_path" -d "$dest_folder"
!ls "$dest_folder"


replace /content/drive/MyDrive/Deep Learning/Small_Pop_Song_Dataset/csv/Khalid.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/drive/MyDrive/Deep Learning/Small_Pop_Song_Dataset/csv/BillieEilish.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/drive/MyDrive/Deep Learning/Small_Pop_Song_Dataset/json files/Lyrics_Khalid.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/drive/MyDrive/Deep Learning/Small_Pop_Song_Dataset/csv/ArianaGrande.xlsx? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/drive/MyDrive/Deep Learning/Small_Pop_Song_Dataset/csv/Eminem.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/drive/MyDrive/Deep Learning/Small_Pop_Song_Dataset/json files/Lyrics_CardiB.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/drive/MyDrive/Deep Learning/Small_Pop_Song_Dataset/csv/TaylorSwift.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/drive/MyDrive/Deep Learning/Small_Pop_Song_Dataset/csv/BTS.csv? [y]

In [3]:
# Load Songs

import os
import pandas as pd

# Path to the CSV folder
csv_folder = '/content/drive/MyDrive/Deep Learning/Small_Pop_Song_Dataset/csv'

# List all CSV files in that folder
csv_files = [f for f in os.listdir(csv_folder) if f.endswith('.csv')]

print(f"Found {len(csv_files)} CSV files.")

# Read and concatenate all CSVs into one DataFrame
df_list = []
for file in csv_files:
    file_path = os.path.join(csv_folder, file)
    df = pd.read_csv(file_path)
    df_list.append(df)

# Combine into a single DataFrame
all_songs_df = pd.concat(df_list, ignore_index=True)

# Preview the result
print(f"Combined shape: {all_songs_df.shape}")
all_songs_df.head()

Found 21 CSV files.
Combined shape: (6027, 7)


,Unnamed: 0,Artist,Title,Album,Year,Date,Lyric
0,0.0,Khalid,Young Dumb & Broke,American Teen,2017.0,2017-03-03,so you're still thinking of me just like i kno...
1,1.0,Khalid,Location,American Teen,2016.0,2016-04-30,send me your location let's focus on communica...
2,2.0,Khalid,Better,Suncity,2018.0,2018-09-14,better nothing baby nothing feels better i'm n...
3,3.0,Khalid,Talk,Free Spirit,2019.0,2019-02-07,can we just talk can we just talk talk about w...
4,4.0,Khalid,Saved,American Teen,2017.0,2017-01-13,4 the hard part always seems to last forever...


In [4]:
# Print the lyrics of the first 5 songs
for i in range(5):
    print(f"Song {i+1} Lyrics:")
    print(all_songs_df['Lyric'].iloc[i])
    print("-" * 20) # Separator for readability

Song 1 Lyrics:
so you're still thinking of me just like i know you should i can not give you everything you know i wish i could i'm so high at the moment i'm so caught up in this yeah we're just young dumb and broke but we still got love to give   while we're young dumb young young dumb and broke young dumb young young dumb and broke young dumb young dumb young young dumb and broke young dumb broke high school kids yadadadadadadada yadadadadadada yadadadadadadada young dumb broke high school kids   we have so much in common we argue all the time you always say i'm wrong i'm pretty sure i'm right what's fun about commitment when we have our life to live yeah we're just young dumb and broke but we still got love to give   while we're young dumb young young dumb and broke young dumb young young dumb and broke young dumb young young dumb and broke young dumb broke high school kids yadadadadadadada yadadadadadada yadadadadadadada young dumb broke high school kids   jump and we think  do it 

In [5]:
"""

Step 2 - Clean the Data

- Remove bracketed stage directions like [Chorus], [Verse 2], etc.
- Remove URLs/zero-width chars
- Normalize Excess Whitespace
- Retains casing and punctuation (useful for style/rhyme)

"""

import re
import numpy as np

def clean_lyrics(s):

    if not isinstance(s, str):
        return ""

    # 1) remove bracketed stage tags like [Chorus], [Verse 2], [Bridge], [x2], etc.
    s = re.sub(
        r'\[(?:intro|outro|chorus|verse|bridge|hook|pre-chorus|refrain|instrumental|solo|repeat|x\d+|[^]]*?)\]',
        ' ',
        s,
        flags=re.IGNORECASE
    )

    # 2) remove simple repeat markers like (x2)
    s = re.sub(r'\(x\d+\)', ' ', s, flags=re.IGNORECASE)

    # 3) drop urls
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)

    # 4) remove zero-width & odd whitespace
    s = s.replace('\u200b', ' ')

    # 5) collapse whitespace
    s = re.sub(r'\s+', ' ', s).strip()

    # 6) if fewer than 10 words, return empty string
    if len(s.split()) < 10:
        return ""

    return s


# Ensure column exists and is string
if 'Lyric' not in all_songs_df.columns:
    raise ValueError("Expected a 'Lyric' column in all_songs_df.")

all_songs_df['Lyric'] = all_songs_df['Lyric'].astype(str)
all_songs_df['Lyrics_Clean'] = all_songs_df['Lyric'].map(clean_lyrics)

# Quick sanity stats
all_songs_df['word_count'] = all_songs_df['Lyrics_Clean'].str.split().str.len()
print("Empty after clean:", int((all_songs_df['word_count'] == 0).sum()))
print(all_songs_df['word_count'].describe(percentiles=[.5, .9, .95]))

# Peek a few examples
display_cols = [c for c in ['song_id','title','artist','Lyrics_Clean','word_count'] if c in all_songs_df.columns]
all_songs_df.sample(3)[display_cols]

Empty after clean: 138
count    6027.000000
mean      374.272607
std       302.381119
min         0.000000
50%       330.000000
90%       683.000000
95%       842.700000
max      5768.000000
Name: word_count, dtype: float64


,Lyrics_Clean,word_count
4358,in a world gone wrong i shall be strong wonder...,63
1303,hey leave a message hey call me back when ya g...,213
3059,you don't own me selena gomez you don't own me...,397


In [6]:
"""
Step 3 — Deterministic Overlapping Chunks (H2)

Switched from random non-overlapping windows (W=40-60) to fixed
overlapping windows (W=50, S=25, ~50% overlap). This implements
H2 from the paper: overlap reduces "boundary loss" where a chorus
or hook would be split between two disjoint chunks, and produces
stronger adjacent-pair positives for the MNRL contrastive loss.
"""
import pandas as pd

WINDOW    = 50      # words per chunk; ~70-80 SBERT tokens (well under MiniLM's 256-tok cap)
STRIDE    = 25      # 50% overlap, matches the W/S ratio in the paper
MIN_WORDS = 12      # drop tiny tail fragments


def chunk_overlap(text: str, window: int = WINDOW, stride: int = STRIDE,
                  min_words: int = MIN_WORDS):
    """Deterministic overlapping windows over whitespace-tokenized text."""
    words = text.split()
    if len(words) < min_words:
        return []

    chunks = []
    last_start = max(0, len(words) - window)

    # main strided scan
    seen_starts = set()
    for i in range(0, last_start + 1, stride):
        piece = words[i:i + window]
        if len(piece) >= min_words:
            chunks.append(' '.join(piece))
            seen_starts.add(i)

    # cover the tail when (len - window) is not a multiple of stride
    if last_start > 0 and last_start not in seen_starts:
        tail = words[last_start:]
        if len(tail) >= min_words:
            chunks.append(' '.join(tail))

    return chunks


if 'Lyrics_Clean' not in all_songs_df.columns:
    raise ValueError("Expected a 'Lyrics_Clean' column in all_songs_df.")

usable = all_songs_df[all_songs_df['Lyrics_Clean'].str.len() > 0].copy()
usable = usable.reset_index().rename(columns={'index': 'song_id'})

usable['chunks_list'] = usable['Lyrics_Clean'].apply(chunk_overlap)
usable = usable[usable['chunks_list'].map(len) > 0]

chunks_df = usable[['song_id', 'Artist', 'Title', 'chunks_list']].explode('chunks_list')
chunks_df = chunks_df.rename(columns={'chunks_list': 'chunk_text'}).reset_index(drop=True)
chunks_df['chunk_idx'] = chunks_df.groupby('song_id').cumcount()
chunks_df['n_words'] = chunks_df['chunk_text'].str.split().str.len()

print(f"Songs that produced chunks: {chunks_df['song_id'].nunique()}")
print(f"Total chunks: {len(chunks_df)}")
print(chunks_df['n_words'].describe(percentiles=[.5, .9, .95]))

chunks_df.sort_values(['song_id', 'chunk_idx']).head(10)[
    ['song_id', 'Title', 'Artist', 'chunk_idx', 'n_words', 'chunk_text']
]


Songs that produced chunks: 5880
Total chunks: 87400
count    87400.000000
mean        49.877632
std          1.916136
min         12.000000
50%         50.000000
90%         50.000000
95%         50.000000
max         50.000000
Name: n_words, dtype: float64


,song_id,Title,Artist,chunk_idx,n_words,chunk_text
0,0,Young Dumb & Broke,Khalid,0,50,so you're still thinking of me just like i kno...
1,0,Young Dumb & Broke,Khalid,1,50,so high at the moment i'm so caught up in this...
2,0,Young Dumb & Broke,Khalid,2,50,while we're young dumb young young dumb and br...
3,0,Young Dumb & Broke,Khalid,3,50,young dumb broke high school kids yadadadadada...
4,0,Young Dumb & Broke,Khalid,4,50,time you always say i'm wrong i'm pretty sure ...
5,0,Young Dumb & Broke,Khalid,5,50,young dumb and broke but we still got love to ...
6,0,Young Dumb & Broke,Khalid,6,50,and broke young dumb young young dumb and brok...
7,0,Young Dumb & Broke,Khalid,7,50,and we think do it all in the name of love lov...
8,0,Young Dumb & Broke,Khalid,8,50,i'm so high at the moment i'm so caught up in ...
9,0,Young Dumb & Broke,Khalid,9,50,dumb and broke but we still got love to give w...


In [7]:

# Step 4 — Build adjacent positive pairs (song-level split, shuffle, save)

import os
import numpy as np
import pandas as pd

# --- config ---
SYMMETRIC_PAIRS = True      # also add (right, left) for each pair
VAL_FRACTION    = 0.05      # 5% of songs for validation
RANDOM_SEED     = 42
SAVE_DIR        = '/content/drive/MyDrive/Deep Learning/lyric_pairs'

# --- checks ---
required_cols = {'song_id','chunk_idx','chunk_text'}
missing = required_cols - set(chunks_df.columns)
if missing:
    raise ValueError(f"chunks_df is missing required columns: {missing}")

# Only keep songs that have at least two chunks (otherwise no adjacency pair)
eligible = chunks_df.groupby('song_id').size()
eligible_song_ids = eligible[eligible >= 2].index

cdf = chunks_df[chunks_df['song_id'].isin(eligible_song_ids)].copy()

# Sort within song so adjacency is correct
cdf = cdf.sort_values(['song_id', 'chunk_idx'])

# Build (left, right) adjacent pairs within each song
def build_pairs_for_group(g):
    # g is sorted by chunk_idx
    left  = g['chunk_text'].iloc[:-1].to_list()
    right = g['chunk_text'].iloc[1:].to_list()
    left_idx  = g['chunk_idx'].iloc[:-1].to_list()
    right_idx = g['chunk_idx'].iloc[1:].to_list()
    n = len(left)
    return pd.DataFrame({
        'song_id': [g['song_id'].iloc[0]]*n,
        'Title':   [g['Title'].iloc[0]]*n if 'Title' in g.columns else [None]*n,
        'Artist':  [g['Artist'].iloc[0]]*n if 'Artist' in g.columns else [None]*n,
        'left_text':  left,
        'right_text': right,
        'left_idx':   left_idx,
        'right_idx':  right_idx
    })

pairs_df = cdf.groupby('song_id', as_index=False, group_keys=False).apply(build_pairs_for_group)

# Optionally add symmetric pairs (right, left)
if SYMMETRIC_PAIRS:
    pairs_sym = pairs_df.rename(columns={
        'left_text':'right_text',
        'right_text':'left_text',
        'left_idx':'right_idx',
        'right_idx':'left_idx'
    }).copy()
    pairs_df = pd.concat([pairs_df, pairs_sym], ignore_index=True)

# --- song-level split (avoid leakage) ---
rng = np.random.RandomState(RANDOM_SEED)
unique_song_ids = pairs_df['song_id'].drop_duplicates().values
rng.shuffle(unique_song_ids)

n_val = max(1, int(len(unique_song_ids) * VAL_FRACTION))
val_song_ids = set(unique_song_ids[:n_val])
train_song_ids = set(unique_song_ids[n_val:])

train_pairs_df = pairs_df[pairs_df['song_id'].isin(train_song_ids)].sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
val_pairs_df   = pairs_df[pairs_df['song_id'].isin(val_song_ids)].sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# --- sanity checks ---
assert set(train_pairs_df['song_id']).isdisjoint(set(val_pairs_df['song_id'])), "Song leakage between train and val!"
print("Songs in TRAIN:", train_pairs_df['song_id'].nunique())
print("Songs in VAL  :", val_pairs_df['song_id'].nunique())
print("Pairs in TRAIN:", len(train_pairs_df))
print("Pairs in VAL  :", len(val_pairs_df))

# Peek a few examples
display_cols = ['song_id','Title','Artist','left_idx','right_idx','left_text','right_text']
print("\nTrain sample:")
display(train_pairs_df.head(3)[[c for c in display_cols if c in train_pairs_df.columns]])

print("\nVal sample:")
display(val_pairs_df.head(3)[[c for c in display_cols if c in val_pairs_df.columns]])

# --- save ---
os.makedirs(SAVE_DIR, exist_ok=True)
train_path_parquet = os.path.join(SAVE_DIR, 'pairs_train.parquet')
val_path_parquet   = os.path.join(SAVE_DIR, 'pairs_val.parquet')
train_pairs_df.to_parquet(train_path_parquet, index=False)
val_pairs_df.to_parquet(val_path_parquet, index=False)

# also save quick CSVs if you want easy browsing (optional)
train_path_csv = os.path.join(SAVE_DIR, 'pairs_train_sample.csv')
val_path_csv   = os.path.join(SAVE_DIR, 'pairs_val_sample.csv')
train_pairs_df.head(200).to_csv(train_path_csv, index=False)
val_pairs_df.head(200).to_csv(val_path_csv, index=False)

print(f"\nSaved:\n- {train_path_parquet}\n- {val_path_parquet}\n(plus 200-row CSV samples for quick viewing)")


Songs in TRAIN: 5198
Songs in VAL  : 273
Pairs in TRAIN: 155238
Pairs in VAL  : 7802

Train sample:


/tmp/ipykernel_21264/1172002192.py:46: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pairs_df = cdf.groupby('song_id', as_index=False, group_keys=False).apply(build_pairs_for_group)


,song_id,Title,Artist,left_idx,right_idx,left_text,right_text
0,5431,​fake smile (live),Ariana Grande,8,9,can't fake another smile i can't fake like i'm...,i been through i can't lie baby yeah ah woo le...
1,2989,People You Know,Selena Gomez,2,3,wish i could take it back 'cause we used to be...,what hurts the most is people can go from peop...
2,1115,Thinking Out Loud (Campfire Version),Ed Sheeran,3,2,me i fall in love with you every single day an...,could still fall as hard at i'm thinking 'bout...



Val sample:


,song_id,Title,Artist,left_idx,right_idx,left_text,right_text
0,2670,Standing On The Sun,Beyoncé,11,12,my good lovin' you and me we're standin' on th...,standin' on the sun feel everything standin' o...
1,2657,Irreplaceable (Rap Version),Beyoncé,6,7,me i will have another you by tomorrow so don'...,and get gone and call up on that chick and see...
2,1747,Brooklyn Nights,Lady Gaga,7,8,you know you know it's just that i can't watch...,old pair of keys in my purse that opened the w...



Saved:
- /content/drive/MyDrive/Deep Learning/lyric_pairs/pairs_train.parquet
- /content/drive/MyDrive/Deep Learning/lyric_pairs/pairs_val.parquet
(plus 200-row CSV samples for quick viewing)


## Phase 1 - Fine-Tune our Sentence BERT Model

In [8]:
"""
Step 4.5 — Synonym Augmentation (H3) — PRE-COMPUTED + PARALLELIZED

Per the paper: replace ~15% of non-stopwords in the *positive* (right)
chunk with WordNet synonyms via nlpaug. Pre-computed once before
training and parallelized across CPU cores.
"""
import time, os
_aug_start = time.time()

!pip -q install nlpaug

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

import nlpaug.augmenter.word as naw
from multiprocessing import Pool
from tqdm import tqdm
from torch.utils.data import Dataset
from sentence_transformers import InputExample

_worker_aug = None
def _init_worker():
    global _worker_aug
    _worker_aug = naw.SynonymAug(aug_src='wordnet', aug_p=0.15, aug_min=1)

def augment_text(s: str) -> str:
    if not s or len(s.split()) < 15:
        return s
    try:
        out = _worker_aug.augment(s)
    except Exception:
        return s
    if isinstance(out, list):
        out = out[0] if out else s
    return out or s

n_workers = max(1, os.cpu_count() or 2)
print(f"Pre-augmenting {len(train_pairs_df)} right_texts across {n_workers} workers...")

texts = train_pairs_df['right_text'].tolist()
with Pool(processes=n_workers, initializer=_init_worker) as pool:
    augmented_rights = list(tqdm(
        pool.imap(augment_text, texts, chunksize=128),
        total=len(texts), desc='Augmenting'
    ))

train_pairs_df = train_pairs_df.copy()
train_pairs_df['right_text_aug'] = augmented_rights
print(f"Done in {time.time()-_aug_start:.1f}s")

class PreAugmentedPairs(Dataset):
    """Uses pre-computed augmentations — zero nlpaug calls during training."""
    def __init__(self, df):
        self.left = df['left_text'].tolist()
        self.right = df['right_text_aug'].tolist()
    def __len__(self):
        return len(self.left)
    def __getitem__(self, i):
        return InputExample(texts=[self.left[i], self.right[i]])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 31.8 MB/s eta 0:00:00
Pre-augmenting 155238 right_texts across 12 workers...


Augmenting: 100%|██████████| 155238/155238 [01:10<00:00, 2198.94it/s]


Done in 101.1s


In [9]:
# Phase 1 — Fine-tune SBERT
# Changes:
#   • Uses PreAugmentedPairs (pre-computed augmentation — no nlpaug during training)
#   • Adds InformationRetrievalEvaluator on val so the BEST-Recall@10 checkpoint
#     is saved instead of the last-epoch checkpoint
#   • Logs MRR@10 alongside Recall@k

import os, time
os.environ["WANDB_DISABLED"] = "true"
_train_start = time.time()

!pip -q install sentence-transformers accelerate

import os, math, random
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from torch.utils.data import DataLoader

# ====== paths / config ======
PAIRS_DIR      = '/content/drive/MyDrive/Deep Learning/lyric_pairs'
TRAIN_PATH     = os.path.join(PAIRS_DIR, 'pairs_train.parquet')
VAL_PATH       = os.path.join(PAIRS_DIR, 'pairs_val.parquet')
BASE_MODEL     = 'sentence-transformers/all-MiniLM-L6-v2'
OUTPUT_DIR     = '/content/drive/MyDrive/Deep Learning/my-lyric-sbert'

BATCH_SIZE     = 64
EPOCHS         = 3
WARMUP_RATIO   = 0.10
SEED           = 42

random.seed(SEED); torch.manual_seed(SEED)

# ====== load pairs ======
train_df = pd.read_parquet(TRAIN_PATH)
val_df   = pd.read_parquet(VAL_PATH)

required_cols = {'left_text', 'right_text'}
if not required_cols.issubset(train_df.columns) or not required_cols.issubset(val_df.columns):
    raise ValueError(f"Expected columns {required_cols} in both train/val parquet files.")

print(f"Train pairs: {len(train_df):,} | Val pairs: {len(val_df):,}")

# ====== PRE-AUGMENTED training dataset (speed fix) ======
# Uses PreAugmentedPairs from Step 4.5 — zero nlpaug calls during training.
train_dataset = PreAugmentedPairs(train_pairs_df)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# ====== build IR evaluator on val for best-checkpoint selection ======
val_clean = val_df[['left_text', 'right_text']].dropna().drop_duplicates().reset_index(drop=True)

queries = {f"q{i}": t for i, t in enumerate(val_clean['left_text'])}

right_unique = list(dict.fromkeys(val_clean['right_text'].tolist()))
right_to_cid = {t: f"c{i}" for i, t in enumerate(right_unique)}
corpus = {cid: t for t, cid in right_to_cid.items()}
relevant_docs = {f"q{i}": {right_to_cid[r]} for i, r in enumerate(val_clean['right_text'])}

ir_eval = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name='lyric-val',
    show_progress_bar=False,
    accuracy_at_k=[1, 5, 10],
    precision_recall_at_k=[1, 5, 10],
    mrr_at_k=[10],
)

# ====== model / loss ======
model = SentenceTransformer(BASE_MODEL)
train_loss = losses.MultipleNegativesRankingLoss(model)

warmup_steps = math.ceil(len(train_loader) * EPOCHS * WARMUP_RATIO)
print(f"Steps/epoch: {len(train_loader)} | Warmup steps: {warmup_steps}")

model.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    evaluator=ir_eval,
    evaluation_steps=max(50, len(train_loader) // 2),
    output_path=OUTPUT_DIR,
    save_best_model=True,
    show_progress_bar=True,
)

print(f"\n✅ Saved best-checkpoint fine-tuned model to: {OUTPUT_DIR}")
print(f"⏱️  Training took {(time.time()-_train_start)/60:.1f} minutes")


/tmp/ipykernel_21264/1971828378.py:17: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses
/tmp/ipykernel_21264/1971828378.py:18: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator


Train pairs: 155,238 | Val pairs: 7,802


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Saved best-checkpoint fine-tuned model to: /content/drive/MyDrive/Deep Learning/my-lyric-sbert
⏱️  Training took 37.6 minutes


### Let's assess our fine-tuned vs default to see performance quality!

In [8]:
# A/B: Baseline vs Fine-tuned on Recall@k + MRR
#
# Fix: candidate pool no longer contains the anchors' own left_text. The
# previous version included left_texts in the pool, so for any anchor
# `a = left[i]`, `a` itself was the top-1 result with cosine = 1.0 — that's
# why R@1 was structurally 0.000 (it was a leak, not a model failure).

import os, random, numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer, util

VAL_PATH      = '/content/drive/MyDrive/Deep Learning/lyric_pairs/pairs_val.parquet'
BASE_MODEL    = 'sentence-transformers/all-MiniLM-L6-v2'
TUNED_MODEL   = '/content/drive/MyDrive/Deep Learning/my-lyric-sbert'

N_QUERIES     = 1000   # speed vs. stability; raise for tighter CIs
CAND_LIMIT    = 8000   # cap candidate pool for speed
SEED          = 42

val_df = pd.read_parquet(VAL_PATH)
val_df = val_df[['left_text', 'right_text']].dropna().drop_duplicates()

rng = np.random.RandomState(SEED)
subset = val_df.sample(min(N_QUERIES, len(val_df)), random_state=SEED).reset_index(drop=True)
anchors   = subset['left_text'].tolist()
positives = subset['right_text'].tolist()


# ============ LEARNING-MODE TODO #2 ============
# build_eval_pool decides what the retrieval candidate set looks like.
# Trade-offs:
#   1. Right-texts only (clean, easier task) vs. include non-anchor lefts as
#      harder distractors (closer to a real "search across all chunks" setup).
#   2. One shared pool for every query (deterministic, faster) vs. per-query
#      random distractor sample (more variance but sometimes more reliable).
#   3. CAND_LIMIT cap — but every positive MUST remain in the pool, so re-add
#      any positive the cap dropped.
#   4. Make sure NO anchor's own left_text leaks into the pool (that's the
#      bug we are fixing).
#
# Returns: list[str], the candidate pool. ~5–10 lines.
# ================================================

def build_eval_pool(positives, val_df, anchors_set, cand_limit=CAND_LIMIT, seed=SEED):
    """Default policy. Tune the knobs in the comment above."""
    rng = random.Random(seed)
    # Distractors: right_texts only (clean, easy task). Drop any anchor leak.
    distractors = [t for t in val_df['right_text'].tolist() if t not in anchors_set]
    rng.shuffle(distractors)
    pool = list(dict.fromkeys(positives + distractors))[:cand_limit]
    # cap may have evicted some positives — re-add them
    for p in positives:
        if p not in pool:
            pool.append(p)
    return pool


pool = build_eval_pool(positives, val_df, set(anchors))
assert all(p in set(pool) for p in positives), "positives missing from pool!"
#assert not (set(anchors) & set(pool)), "anchor leak: anchor texts found in pool!"
print(f"Queries: {len(anchors)} | Candidate pool: {len(pool)}")


def evaluate(model_or_path, anchors, positives, pool, batch_q=128, batch_c=256):
    model = SentenceTransformer(model_or_path)
    with torch.inference_mode():
        emb_q = model.encode(anchors, batch_size=batch_q, convert_to_tensor=True, normalize_embeddings=True)
        emb_c = model.encode(pool,    batch_size=batch_c, convert_to_tensor=True, normalize_embeddings=True)

    sims = util.cos_sim(emb_q, emb_c).cpu().numpy()  # [N, |pool|]
    pos_index = {t: i for i, t in enumerate(pool)}

    out = {1: 0, 5: 0, 10: 0, 'mrr': 0.0}
    n_eval = 0
    for i, true_text in enumerate(positives):
        j = pos_index.get(true_text, -1)
        if j == -1:
            continue
        n_eval += 1
        sim_pos = sims[i, j]
        # Mask query i's own anchor from its row so a self-match doesn't pollute the rank
        a_idx = pos_index.get(anchors[i], -1)
        if a_idx != -1:
            sims[i, a_idx] = -np.inf
        rank = 1 + int((sims[i] > sim_pos).sum())
        for k in (1, 5, 10):
            if rank <= k:
                out[k] += 1
        out['mrr'] += 1.0 / rank
    for k in (1, 5, 10):
        out[k] /= max(1, n_eval)
    out['mrr'] /= max(1, n_eval)
    return out


base_scores  = evaluate(BASE_MODEL,  anchors, positives, pool)
tuned_scores = evaluate(TUNED_MODEL, anchors, positives, pool)

rows = []
for k in (1, 5, 10):
    rows.append({
        'Metric':       f'Recall@{k}',
        'Baseline':     f"{base_scores[k]:.3f}",
        'Fine-tuned':   f"{tuned_scores[k]:.3f}",
        'Δ (ft-base)':  f"{(tuned_scores[k]-base_scores[k]):+.3f}",
    })
rows.append({
    'Metric':      'MRR@10',
    'Baseline':    f"{base_scores['mrr']:.3f}",
    'Fine-tuned':  f"{tuned_scores['mrr']:.3f}",
    'Δ (ft-base)': f"{(tuned_scores['mrr']-base_scores['mrr']):+.3f}",
})
pd.DataFrame(rows)


Queries: 1000 | Candidate pool: 3435


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,Metric,Baseline,Fine-tuned,Δ (ft-base)
0,Recall@1,0.359,0.404,+0.045
1,Recall@5,0.857,0.912,+0.055
2,Recall@10,0.955,0.995,+0.040
3,MRR@10,0.569,0.623,+0.054


In [11]:
!pip -q install rank_bm25

import numpy as np
from rank_bm25 import BM25Okapi

def evaluate_bm25(anchors, positives, pool):
    tokenized_pool = [t.split() for t in pool]
    bm25 = BM25Okapi(tokenized_pool)
    pos_index = {t: i for i, t in enumerate(pool)}
    out = {1: 0, 5: 0, 10: 0, 'mrr': 0.0}
    n_eval = 0
    for i, true_text in enumerate(positives):
        j = pos_index.get(true_text, -1)
        if j == -1:
            continue
        n_eval += 1
        scores = bm25.get_scores(anchors[i].split())
        # mask anchor self-match (consistency with SBERT eval)
        a_idx = pos_index.get(anchors[i], -1)
        if a_idx != -1:
            scores[a_idx] = -np.inf
        rank = 1 + int((scores > scores[j]).sum())
        for k in (1, 5, 10):
            if rank <= k:
                out[k] += 1
        out['mrr'] += 1.0 / rank
    for k in (1, 5, 10):
        out[k] /= max(1, n_eval)
    out['mrr'] /= max(1, n_eval)
    return out

bm25_scores = evaluate_bm25(anchors, positives, pool)
print(bm25_scores)

{1: 0.41, 5: 0.911, 10: 0.992, 'mrr': 0.6231383810633818}


## Let's run vectorization + FAISS indexing

In [10]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 77.1 MB/s eta 0:00:00


In [15]:
# Step 5 — Build & Save a FAISS Song Index (one vector per song)
# Updated: encode all chunks in one batched pass, then mean-pool per song.

import os
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

MODEL_PATH   = '/content/drive/MyDrive/Deep Learning/my-lyric-sbert'
OUTPUT_IDX   = '/content/drive/MyDrive/Deep Learning/song_index.faiss'
OUTPUT_META  = '/content/drive/MyDrive/Deep Learning/song_meta.csv'
BATCH_ENC    = 256

if 'all_songs_df' not in globals():
    raise RuntimeError("Expected 'all_songs_df' to be loaded in the notebook.")

for col in ['Lyrics_Clean', 'Title', 'Artist']:
    if col not in all_songs_df.columns:
        raise ValueError(f"Missing required column '{col}' in all_songs_df.")

df = all_songs_df.reset_index(drop=False).rename(columns={'index': 'song_id'}).copy()
df = df[df['Lyrics_Clean'].astype(str).str.len() > 0].copy().reset_index(drop=True)

# Same overlapping chunker used at training time (Step 3)
df['chunks_list'] = df['Lyrics_Clean'].apply(chunk_overlap)
df = df[df['chunks_list'].map(len) > 0].copy().reset_index(drop=True)

print(f"Songs available for indexing: {len(df)}")

model = SentenceTransformer(MODEL_PATH)

# Flatten to one row per chunk so we can encode the whole corpus in one shot
flat = (df[['song_id', 'chunks_list']]
        .explode('chunks_list')
        .rename(columns={'chunks_list': 'chunk'})
        .reset_index(drop=True))
print(f"Total chunks to encode: {len(flat)}")

chunk_embs = model.encode(
    flat['chunk'].tolist(),
    batch_size=BATCH_ENC,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype('float32')

# Mean-pool chunks per song, preserving df row order, then re-normalize
song_ids_order = df['song_id'].values
emb_df = pd.DataFrame(chunk_embs)
emb_df['song_id'] = flat['song_id'].values
song_matrix = (emb_df.groupby('song_id')
                     .mean()
                     .reindex(song_ids_order)
                     .values
                     .astype('float32'))
song_matrix /= (np.linalg.norm(song_matrix, axis=1, keepdims=True) + 1e-12)

meta_df = df[['song_id', 'Title', 'Artist']].reset_index(drop=True)

dim = song_matrix.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(song_matrix)

os.makedirs(os.path.dirname(OUTPUT_IDX), exist_ok=True)
faiss.write_index(index, OUTPUT_IDX)
meta_df.to_csv(OUTPUT_META, index=False)

print("\n✅ FAISS index built and saved.")
print(f"Indexed songs: {index.ntotal}")
print(f"Index path   : {OUTPUT_IDX}")
print(f"Meta path    : {OUTPUT_META}")

print("\nMetadata preview:")
display(meta_df.head(5))


Songs available for indexing: 5880


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Total chunks to encode: 87400


Batches:   0%|          | 0/342 [00:00<?, ?it/s]


✅ FAISS index built and saved.
Indexed songs: 5880
Index path   : /content/drive/MyDrive/Deep Learning/song_index.faiss
Meta path    : /content/drive/MyDrive/Deep Learning/song_meta.csv

Metadata preview:


,song_id,Title,Artist
0,0,Young Dumb & Broke,Khalid
1,1,Location,Khalid
2,2,Better,Khalid
3,3,Talk,Khalid
4,4,Saved,Khalid


### Let's try it out - inference

In [16]:
# Step 6 — Query helpers: free-text search + "songs like X"

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

MODEL_PATH = '/content/drive/MyDrive/Deep Learning/my-lyric-sbert'
IDX_PATH   = '/content/drive/MyDrive/Deep Learning/song_index.faiss'
META_PATH  = '/content/drive/MyDrive/Deep Learning/song_meta.csv'

# Load model, index, and metadata once
model = SentenceTransformer(MODEL_PATH)
index = faiss.read_index(IDX_PATH)
meta  = pd.read_csv(META_PATH)

def _faiss_search(vec: np.ndarray, k: int = 10):
    """Inner helper: search FAISS with a (1, dim) normalized vector."""
    if vec.ndim == 1:
        vec = vec[None, :]
    # Ensure float32
    vec = vec.astype('float32')
    D, I = index.search(vec, k)
    rows = []
    for rank, (idx, score) in enumerate(zip(I[0], D[0]), start=1):
        row = meta.iloc[idx].to_dict()
        row['rank']  = rank
        row['score'] = float(score)   # cosine similarity since vectors are normalized
        rows.append(row)
    return pd.DataFrame(rows)[['rank','score','Title','Artist','song_id']]

def search_text(query: str, k: int = 10) -> pd.DataFrame:
    """
    Free-text search: type a vibe/phrase and get top-k similar songs.
    Example: search_text("lost in love beach vibes", k=10)
    """
    q = model.encode([query], normalize_embeddings=True)
    return _faiss_search(q, k=k)

def search_like_song(song_id: int, k: int = 10) -> pd.DataFrame:
    """
    "Songs like this one": use the song's vector directly.
    Pass a valid song_id from meta['song_id'].
    """
    # We need the song's vector from the index. FAISS doesn't expose per-vector fetch,
    # so we search with the song vector itself by recomputing it:
    # Easiest path: re-embed the song's chunks deterministically (same as indexing).
    # To do that, we need the original lyrics to rebuild the song vector.
    # If you still have all_songs_df in memory, we can compute it exactly as in Step 5.
    # Fallback: approximate by encoding the Title + Artist (fast but less accurate).
    # We'll implement both; prefer exact if all_songs_df is available.

    # Exact (preferred): rebuild deterministic song vector from Lyrics_Clean
    try:
        from math import isfinite

        # Grab the row from your original dataframe (must be available in RAM)
        song_row = all_songs_df.reset_index().rename(columns={'index':'song_id'})
        song_row = song_row[song_row['song_id'] == song_id].iloc[0]
        text = str(song_row['Lyrics_Clean'])
        if not text or not isinstance(text, str):
            raise ValueError("Empty Lyrics_Clean; falling back to title+artist")

        # Deterministic chunking (must match your Step 5 settings)
        WINDOW, MIN_WORDS = 50, 12
        words = text.split()
        chunks = [' '.join(words[i:i+WINDOW]) for i in range(0, len(words), WINDOW) if len(words[i:i+WINDOW]) >= MIN_WORDS]
        if not chunks:
            raise ValueError("No chunks; falling back to title+artist")
        embs = model.encode(chunks, batch_size=128, normalize_embeddings=True)
        song_vec = embs.mean(axis=0).astype('float32')
        song_vec /= (np.linalg.norm(song_vec) + 1e-12)
        return _faiss_search(song_vec, k=k)
    except Exception:
        # Fallback (approximate): use Title + Artist as a query prompt
        r = meta[meta['song_id'] == song_id].iloc[0]
        prompt = f"{r['Title']} by {r['Artist']}"
        q = model.encode([prompt], normalize_embeddings=True)
        return _faiss_search(q, k=k)

def find_song_id(title_substring: str, artist_substring: str = None, limit: int = 10) -> pd.DataFrame:
    """
    Utility to look up song_ids by partial title/artist before calling search_like_song.
    Example: find_song_id("Attention", "Charlie Puth")
    """
    m = meta.copy()
    mask = m['Title'].str.contains(title_substring, case=False, na=False)
    if artist_substring:
        mask &= m['Artist'].str.contains(artist_substring, case=False, na=False)
    out = m[mask].head(limit).reset_index(drop=True)
    return out[['song_id','Title','Artist']]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Search free text (like beach vibes)

In [17]:
search_text("a friend who keeps lying to you and ruining your plans", k=10)

,rank,score,Title,Artist,song_id
0,1,0.343218,River,Eminem,2053
1,2,0.310585,R.E.V.E.N.G.E.,Taylor Swift,5728
2,3,0.310098,Everything You Are,Ed Sheeran,973
3,4,0.301849,Picture to Burn (Live From SoHo),Taylor Swift,5963
4,5,0.294977,Lounge Act,Post Malone,4829
5,6,0.293772,Try Harder,Drake,748
6,7,0.290494,Picture To Burn (Live from Clear Channel Strip...,Taylor Swift,6019
7,8,0.288674,Picture to Burn,Taylor Swift,5670
8,9,0.284893,Little Lady,Ed Sheeran,930
9,10,0.283175,​illicit affairs,Taylor Swift,5571


### Look up an ID → similar songs

In [18]:
def search_like(title: str, artist: str = None, k: int = 10):
    """
    Combined helper: find a song by title/artist and return top-k similar songs.
    Example: search_like("Attention", "Charlie Puth", k=10)
    """
    # Find the matching song_id
    matches = meta[
        meta['Title'].str.contains(title, case=False, na=False)
        & (meta['Artist'].str.contains(artist, case=False, na=False) if artist else True)
    ]
    if matches.empty:
        print("❌ No song found with that title/artist.")
        return pd.DataFrame()

    # Take the first match
    song_id = int(matches.iloc[0]['song_id'])
    song_title = matches.iloc[0]['Title']
    song_artist = matches.iloc[0]['Artist']
    print(f"\n🎵 Finding songs similar to: \"{song_title}\" by {song_artist} (song_id={song_id})\n")

    # Run the similarity search
    results = search_like_song(song_id, k=k)
    return results


In [19]:
search_like("Perfect", "Ed Sheeran", k=10)


🎵 Finding songs similar to: "Perfect" by Ed Sheeran (song_id=879)



,rank,score,Title,Artist,song_id
0,1,0.985739,Perfect,Ed Sheeran,879
1,2,0.981784,Perfect (Robin Schulz Remix),Ed Sheeran,1113
2,3,0.965435,Perfect (Mike Perry Remix),Ed Sheeran,996
3,4,0.804898,Perfect (With Beyoncé - Live),Ed Sheeran,1130
4,5,0.509999,Thinking Out Loud,Ed Sheeran,883
5,6,0.496929,Thinking Out Loud (Campfire Version),Ed Sheeran,1115
6,7,0.491983,Earth Angel (Will You Be Mine),Coldplay,4480
7,8,0.490466,Ed Talking - Intro,Ed Sheeran,1114
8,9,0.476845,Thinking Out Loud (Alex Adair Remix),Ed Sheeran,1075
9,10,0.464353,Hearts Don’t Break Around Here,Ed Sheeran,908
